# 07 - Runtime Prediction Test

This notebook validates the runtime inference path for Healix using the same DDXPlus-derived feature names and exported model artifacts produced during training.

The goal is to demonstrate the final inference pipeline:

ClinicalRecord
        ↓
Feature Encoder
        ↓
XGBoost Model
        ↓
predict_proba()


In [ ]:
from pathlib import Path
import ast
import json
import re
import joblib
import numpy as np
import pandas as pd
from typing import List

from app.domain.clinical_record import ClinicalRecord

# Locate repo root
repo_root = Path(r"c:\healix-ai-medical-assistant")
raw_path = repo_root / "app" / "data" / "raw" / "ddxplus" / "train.csv"

if not raw_path.exists():
    raise FileNotFoundError(f"Could not locate the raw DDXPlus dataset: {raw_path}")

print("Repository root:", repo_root)
print("Raw dataset:", raw_path)

In [ ]:
# Load the exact artifacts created during training
feature_names = json.loads((repo_root / "models" / "feature_names.json").read_text(encoding="utf-8"))
label_encoder = joblib.load(repo_root / "models" / "label_encoder.joblib")
model = joblib.load(repo_root / "models" / "xgboost_model.pkl")

print("Loaded feature names:", len(feature_names))
print("Loaded label encoder classes:", len(label_encoder.classes_))
print("Model type:", type(model).__name__)

In [ ]:
# Normalize and map DDXPlus evidence tokens to the exact exported feature names

def split_token(token: str) -> str:
    return str(token).split("_@_", 1)[0]


def normalize_symptom(value: str) -> str:
    if value is None:
        return ""
    text = str(value).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def parse_evidences(value):
    text = "[]" if pd.isna(value) else str(value)
    try:
        tokens = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        tokens = []
    return [split_token(token) for token in tokens]


def build_feature_vector_from_clinical_record(record: ClinicalRecord, evidence_tokens: List[str], feature_names: List[str]) -> np.ndarray:
    """Build the same binary evidence vector used during training."""
    vector = np.zeros(len(feature_names), dtype=float)

    if "AGE" in feature_names and getattr(record, "chief_complaint", None) is not None:
        age_index = feature_names.index("AGE")
        vector[age_index] = float(getattr(record, "age", 0) or 0)

    if "SEX_M" in feature_names:
        sex_index = feature_names.index("SEX_M")
        vector[sex_index] = 1.0 if str(getattr(record, "gender", "M")).strip().upper() == "M" else 0.0

    present = {split_token(token) for token in evidence_tokens if str(token).strip()}
    for idx, feature_name in enumerate(feature_names):
        if not feature_name.startswith("E_"):
            continue
        symptom_token = feature_name.replace("E_", "", 1)
        if symptom_token in present:
            vector[idx] = 1.0

    return vector


def predict_top_k(record: ClinicalRecord, evidence_tokens: List[str], top_k: int = 5):
    vector = build_feature_vector_from_clinical_record(record, evidence_tokens, feature_names)
    proba = model.predict_proba(vector.reshape(1, -1))[0]
    decoded_labels = label_encoder.inverse_transform(np.arange(len(proba)))
    ranking = sorted(zip(decoded_labels, proba), key=lambda item: item[1], reverse=True)[:top_k]
    return vector, ranking

## Example patient cases

The next cell simulates 10 Healix-style samples from the DDXPlus raw split. Each example is converted into the same feature vector used during training and then ranked by the exported XGBoost model.

In [ ]:
# 10 representative patient examples from the raw DDXPlus split
raw_df = pd.read_csv(raw_path, usecols=["AGE", "SEX", "PATHOLOGY", "EVIDENCES", "INITIAL_EVIDENCE"], nrows=10)
raw_df["evidence_tokens"] = raw_df["EVIDENCES"].apply(parse_evidences)

rows = []
for index, row in raw_df.iterrows():
    record = ClinicalRecord(
        chief_complaint=str(row["INITIAL_EVIDENCE"]),
        severity="moderate",
        duration="unknown",
        body_location="unknown",
        medications=[],
        allergies=[],
        chronic_conditions=[],
        family_history=[],
        missing_fields=[],
    )
    # Attach the demographic fields dynamically for the notebook simulation.
    record.age = float(row["AGE"])
    record.gender = str(row["SEX"])

    vector, ranking = predict_top_k(record, row["evidence_tokens"], top_k=5)
    rows.append(
        {
            "case_id": f"P{index + 1:02d}",
            "pathology": row["PATHOLOGY"],
            "input_evidence_tokens": row["evidence_tokens"],
            "feature_vector": vector.tolist(),
            "top_predictions": ranking,
        }
    )

results_df = pd.DataFrame(rows)
results_df.head()

In [ ]:
# Expand top predictions into a presentation-friendly table
expanded_rows = []
for _, row in results_df.iterrows():
    for disease, probability in row["top_predictions"]:
        expanded_rows.append(
            {
                "case_id": row["case_id"],
                "true_pathology": row["pathology"],
                "input_evidence_tokens": ", ".join(row["input_evidence_tokens"][:12]),
                "disease": disease,
                "probability": round(float(probability), 4),
            }
        )

ranking_df = pd.DataFrame(expanded_rows)
print(ranking_df.head(25).to_string(index=False))

In [ ]:
# Save a report for presentation and future inspection
reports_dir = repo_root / "training" / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
ranking_df.to_csv(reports_dir / "runtime_prediction_examples.csv", index=False)
print("Saved runtime prediction examples to:", reports_dir / "runtime_prediction_examples.csv")

## Interpretation

- Each example is converted into a binary evidence vector using the same feature names exported from the training notebook.
- The exported XGBoost model returns class probabilities through `predict_proba()`.
- The output is intentionally framed as a differential diagnosis / possible conditions ranking rather than a final diagnosis.